In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

In [ ]:
from marine_qc import (
    duplicate_check,
    flag_duplicates,
    get_duplicates,
    remove_duplicates,
)

In [ ]:
from data import get_advanced_data

In [ ]:
import logging


logging.getLogger("splink").setLevel(logging.ERROR)
logging.getLogger("splink.internals").setLevel(logging.ERROR)

# How to use the duplicate checker to detect possibly duplicated observations

The ``marine_qc`` toolbox provides some checks to detect possibly duplicated observations. Creating some test dataset we can show how to use them on "real" data. There are several checks that are shown here:

* a duplicate check to detect possibly duplicated observations
* a function that gets potentially duplicated observations and returns the corresponding detected duplicate matches for each observation
* a function that flags potentially duplicated observations and returns the duplicate flags for each observation
* a function that removes potentially duplicated observations and returns the input data with duplicates excluded

For more information of all available QC checks on individual reports see [the overview of QC functions to detect possibly duplicated observations](https://marine-qc.readthedocs.io/en/latest/overview_dup.html).

The test dataset we provide is a pandas.DataFrame including several individual marine reports.

In [ ]:
data = get_advanced_data()
data

We have several different marine reports that include six different parameters (`lat`, `lon`, `date`, `sst` (sea-surface temperature), `vsi` (ship speed), and `dsi` (ship heading)).

First of all, we detect possibly duplicated observations. This function returns a `marine_qc.duplicate_checker.duplicates.DupDetect` class that contains all information needed for the other duplicate functions. 

## Use the default settings of the duplicate checker

To introduce the duplicate check functions, we work with the default settings. 

> **Note:**
> 
> By default, the duplicate check comprises six kinds of observational values:
> 
> * `station_id`: Any kind of idnetifier of the observing station (e.g. ship name, WIGOS Station Identifier).
> * `lat`: Latitude of the observing station in degrees.
> * `lon`: Longitude of the observing station in degrees.
> * `date`: Time stamp of the obersing station (datetime object or datetime-formatted string).
> * `vsi`: Speed of the observing station in kilometers per hour.
> * `dsi`: Course of the observing station in degrees.
> 
> By default, duplicate observations are detected according to the following criteria:
> 
> * `station_id`: values must match exactly for observations to be considered duplicates.
> * `lat`: values may differ by up to 0.11 degrees for observations to be considered duplicates.
> * `lon`: values may differ by up to 0.11 degrees for observations to be considered duplicates.
> * `date`: values may differ by up to 1 minute for observations to be considered duplicates.
> * `vsi`: values may differ by up to 0.09 kilometers per hour for observations to be considered duplicates.
> * `dsi`: values may differ by up to 0.9 degrees for observations to be considered duplicates.

Later, we will see how to add extra kinds of observational values and their corresponding criteria.

Firstly, we run the `duplicate_check`. The result can be used to run the other duplicate check functions. Alternatively, the functions can be run directly with the data itself. If so, `duplicate_check` is run internally.

In [ ]:
detected = duplicate_check(data=data)

In [ ]:
dups = get_duplicates(
    detected=detected,
)
pd.DataFrame({**data, "duplicates": dups})

As we can see, the following duplicates has been detected:

In [ ]:
data.loc[["B", "H"]]

We expect these to be detected as duplicates since:

* `station_id`: do match exactly
* `lon`: do match exactly
* `lat`: do match exactly
* `date`: differ less than 1 minute
* `vsi`: do match exactly
* `dsi`: do match exactly

In [ ]:
data.loc[["C", "J"]]

We expect these to be detected as duplicates since:

* `station_id`: do match exactly
* `lon`: do match exactly
* `lat`: do match exactly
* `date`: do match exactly
* `vsi`: are both `NaN`
* `dsi`: are both `NaN` 

In [ ]:
data.loc[["D", "L", "N"]]

We expect these to be detected as duplicates since:

* `station_id`: do match exactly
* `lon`: differ less than 0.11 degrees
* `lat`: differ less than 0.11 degrees
* `date`: do match exactly
* `vsi`: do match exactly
* `dsi`: do match exactly

In [ ]:
data.loc[["E", "F", "Q"]]

We expect these to be detected as duplicates since:

* `station_id`: do match exactly
* `lon`: differ less than 0.11 degrees
* `lat`: differ less than 0.11 degrees
* `date`: do match exactly
* `vsi`: do match exactly
* `dsi`: do match exactly

Now, we can flag the duplicates. The flagging scheme is:

* `0`: unique observation
* `1`: best duplicate
* `3`: worst duplicate

In [ ]:
flags = flag_duplicates(detected=detected)
pd.DataFrame({**data, "duplicate_flag": flags})

Optionally, we can remove the duplicated observations.

In [ ]:
series = remove_duplicates(detected=detected)
pd.DataFrame(series).T

## Manipulate the settings of the duplicate checker

Here are some more examples of how to use the duplicate checker with user-specific settings.

* `ignore_entires`: Ignore specific values for selected columns during comparison.
* `ignore_nan_both`: For selected columns, consider two observations as duplicates whenever both compared value are NaN. By default, this is true for all columns.
* `ignore_nan_either`: For selected columns, consider two observations as duplicates whenever either compared value is NaN.
* `ignore_columns`: Column names to exclude entirely from duplicate detection.
* `offsets`: Override comparison offsets for selected columns.
* `compare_level_libraries`: Override comparison levels for selected columns. For more information see [Splink: Comparison Level Library](https://moj-analytical-services.github.io/splink/api_docs/comparison_level_library.html).

### `ignore_entries`

If `station_id` of two observations is "AA" or "BB", they should be detected as duplicates.

In [ ]:
dups = get_duplicates(
    data=data,
    ignore_entries={"station_id": ["AA", "BB"]},
)
pd.DataFrame({**data, "duplicates": dups})

In addition to the default example, we get the following duplicates:

In [ ]:
data.loc[["D", "L", "M", "N", "O"]]

"O" is detected as duplicate as expected. 

<div style="border-left: 4px solid #d9534f; background-color: #f8d7da; color: #721c24; padding: 10px; margin: 10px 0;">
<strong>❌ Error:</strong> Unfortunately, "M" is detected as duplicate as well however the `station_id` is "AB".
</div>

### `ignore_nan_both`

Two observations are considered as duplicates whenever both compared value are NaN only for selected column `vsi`.

In [ ]:
dups = get_duplicates(
    data=data,
    ignore_nan_both="vsi",
)
pd.DataFrame({**data, "duplicates": dups})

In comparison with the default example "C" and "J" are not considered as duplicates since `dsi` has NaN values:

In [ ]:
data.loc[["C", "J"]]

### `ignore_nan_either`

For `vsi`, consider two observations as duplicates whenever either compared value is NaN.

In [ ]:
dups = get_duplicates(
    data=data,
    ignore_nan_either="vsi",
)
pd.DataFrame({**data, "duplicates": dups})

The duplicate group ("B", "H") is extended by ("U", "S").

In [ ]:
data.loc[["B", "H", "U", "S"]]

<div style="border-left: 4px solid #f0ad4e; background-color: #fcf8e3; padding: 10px; margin: 10px 0;">
<strong>Warning:</strong> 

Here we can see, that `ignore_nan_either` can lead to misleading duplicate chains:

* ("B", "H") is obviously a duplicate pair.
* ("H", "U") is a duplicate pair as well since we ignore NaNs for `vsi`.
* ("U", "S") is a duplicate pair as well since we ignore NaNs for `vsi`.

This leads to the following duplicate chain:

"B" -> "H" -> "U" -> "S"

But obviously, ("B", "S") is not a duplicate pair since `vsi` differs more than 0.11 degrees.

</div> 

### `ignore_columns`

We exclude `station_id` entirely from duplicate detection.

In [ ]:
dups = get_duplicates(
    data=data,
    ignore_columns="station_id",
)
pd.DataFrame({**data, "duplicates": dups})

We extend one duplicate group from the default example since we ignore `station_id` completely.

In [ ]:
data.loc[["D", "L", "M", "N", "O"]]

### `offsets`

We now consider two observations as duplicates if:

* `lat` differs up to 1.0 degrees
* `lon` differs up to 1.0 degrees
* `date` differs up to 360 seconds (1 hour)

In [ ]:
dups = get_duplicates(
    data=data,
    offsets={"lat": 1.0, "lon": 1.0, "date": 360},
)
pd.DataFrame({**data, "duplicates": dups})

We expand two duplicate groups:

In [ ]:
data.loc[["D", "K", "L", "N"]]

In [ ]:
data.loc[["E", "F", "P", "Q"]]

### `compare_level_libraries`

Finally, we add `sst` with an offset of 1K to the criteria list.

In [ ]:
dups = get_duplicates(
    data=data,
    compare_level_libraries={"sst": "AbsoluteDifferenceLevel"},
    offsets={"sst": 1.0},
)
pd.DataFrame({**data, "duplicates": dups})

Now ("B", "H") is no duplicate pair anymore since `sst` differs more than 1 Kelvin.

In [ ]:
data.loc[["B", "H"]]